# Gold — Incremental Loader

Watermark-driven incremental load from `bronze.orders` → `silver.orders_incremental` with configurable lookback for late data.

Runs **twice** to show initial watermark progression, then a no-op second pass.

In [ ]:
import sys
import importlib

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
REPO_ROOT = notebook_path.split("/notebooks/")[0]
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import src.gold.incremental_loader as inc_module
importlib.reload(inc_module)

from src.gold.incremental_loader import (
    IncrementalLoadConfig,
    ensure_watermark_table,
    get_watermark,
    run_incremental_load,
    WATERMARK_TABLE,
)
from src.ingestion.idempotent_loader import save_run_report_to_volume

config = IncrementalLoadConfig(lookback_hours=24)
ensure_watermark_table(spark, config.watermark_table)
print("Loaded from:", inc_module.__file__)
print("Watermark table:", WATERMARK_TABLE)

## Run 1 — initial incremental load

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {config.target_table}")
spark.sql(
    f"DELETE FROM {config.watermark_table} WHERE pipeline_id = '{config.pipeline_id}'"
)

run1 = run_incremental_load(spark, config)
print(run1)
display(spark.table(config.watermark_table).filter(f"pipeline_id = '{config.pipeline_id}'"))

## Run 2 — idempotent re-run (expect 0 new records)

In [ ]:
run2 = run_incremental_load(spark, config)
print(run2)
print("Target rows:", spark.table(config.target_table).count())

In [ ]:
import json

report = {
    "task": "gold_incremental_loader",
    "watermark_table": config.watermark_table,
    "pipeline_id": config.pipeline_id,
    "run1_initial_load": run1,
    "run2_idempotent_rerun": run2,
    "watermark_entry": [
        row.asDict()
        for row in spark.table(config.watermark_table)
        .filter(f"pipeline_id = '{config.pipeline_id}'")
        .orderBy("last_run_at")
        .collect()
    ],
}
print(json.dumps(report, indent=2, default=str))
try:
    save_run_report_to_volume(
        report,
        dbutils,
        "/Volumes/globalmart/metadata/run_reports/gold_incremental_loader.json",
    )
except Exception as exc:
    print("Volume save skipped:", exc)